#### Note: This colab only contains short discussion/highlights of task done. Please visit the google document for my detailed writeup:
Google document link:

### TASK 1: Understanding MaxText Data format

To complete task 1 I explored files inside folder `maxtext/input_pipeline` , especially `input_pipeline_inferace.py`.

Here are the dataset formats supported by MaxText:

- `synthetic dataset`: fake dataset (token ids) batches are created in memory, rather than loading any dataset. Used for bechmarkiing hardware throughput purposes etc.

- `grain`: you can load the dataset using following file types:    
  - `arrayrecord`: ArrayRecord file format supports parallel read, write, and random access by record index.

  - `parquet`: Apache parquet format - open-source columnar storage format.


  - `tfrecord`: TensorFlow's standard format for storing a sequence of binary records.




### TASK 2:



To complete task 2 let's first look at the resources available.

In [2]:

import os, psutil, platform

print("=== Hardware Info ===")
print(f"CPU cores: {os.cpu_count()}")
ram = psutil.virtual_memory()
print(f"Total RAM: {ram.total / (1024**3):.2f} GB")
print(f"Available RAM: {ram.available / (1024**3):.2f} GB")
print(f"Platform: {platform.platform()}")

# Check JAX backend
import jax
print(f"\nJAX version: {jax.__version__}")
print(f"JAX backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")

=== Hardware Info ===
CPU cores: 2
Total RAM: 12.67 GB
Available RAM: 11.45 GB
Platform: Linux-6.6.122+-x86_64-with-glibc2.35

JAX version: 0.10.2
JAX backend: gpu
Devices: [CudaDevice(id=0)]


Let's answer the most important question: how much memory is occupied when we train the model and what actually require memory ?

Training requires forward and backward pass:
this is what consume memory:
- Model parameters
- gradients
- optimizer states
- also one thing to note: activations stored from forward pass for backward pass also requires memory.

What also matters is if your model parameter's precision format is in FP32 (32 bit), FP16 (16 bits) or is it quantized (8 bit, 4bit)

For optimizer states, usually it is in fp32

In [3]:
!git clone https://github.com/AI-Hypercomputer/maxtext.git
%cd maxtext


fatal: destination path 'maxtext' already exists and is not an empty directory.
/content/maxtext


In [4]:
!uv venv --python 3.12 --seed exp
!source exp/bin/activate

Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment with seed packages at: exp
? A virtual environment already exists at `exp`. Do you want to replace it? [y/n] › yes

✔ A virtual environment already exists at `exp`. Do you want to replace it? · yes
 + pip==26.1.2
Activate with: source exp/bin/activate


In [5]:
!pip install -r src/dependencies/requirements/generated_requirements/cuda12-requirements.txt
!pip install -e .

Ignoring nest-asyncio: markers 'sys_platform == "win32"' don't match your environment
Ignoring tzdata: markers 'sys_platform == "emscripten" or sys_platform == "win32"' don't match your environment
  Using cached transformer_engine-2.16.0-py3-none-any.whl.metadata (26 kB)
Using cached transformer_engine-2.16.0-py3-none-any.whl (912 kB)
Obtaining file:///content/maxtext
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for maxtext (pyproject.toml) ... done
  Created wheel for maxtext: filename=maxtext-0.2.3-py3-none-any.whl size=19412 sha256=4f76513a610e53efcba7d508aac5812bc1c89366a7ccae1c2ef654d71bcbefb3
  Stored in directory: /tmp/pip-ephem-wheel-cache-4k0x856j/wheels/6c/d1/14/ccb96d070c3f58903f006ad2bc5a7b92d25df031b495676b7b
Successfully built maxtext
  Attempting u

In [6]:
# """
# uninstalling existing transformer engine because it was trying to load incompatible cuDNN:
# more details:
# there is cuda 12.9 but cuDNN (cuda deep neural network) .so files on the system are of mixed versions like pre-installed cuDNN 9.x build
# The 2 .so files needs to be from same cuDNN release.
# those 2 files
# 1)libcudnn_graph.so.9
# 2)libcudnn_heuristic.so.9
# """
!pip uninstall transformer-engine -y


Found existing installation: transformer_engine 2.16.0
Uninstalling transformer_engine-2.16.0:
  Successfully uninstalled transformer_engine-2.16.0


In [1]:
import os
os.environ['XLA_FLAGS'] = '--xla_gpu_autotune_level=0'
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'


In [2]:
# # ## Using attention=dot_product (by default it is autoselected which might choose flash attention). flash attention kernels probably
# # ## might be written in triton and Triton native GPU kernel execution failed on Turing architecture (T4) and requires ampere architecture
# # ## hence using attention=dot_product
# Set CUDA Allocator to handle fragmentation dynamically
!export TF_GPU_ALLOCATOR=cuda_malloc_async
!python3 -m maxtext.trainers.pre_train.train \
  /content/maxtext/src/maxtext/configs/base.yml \
  hardware=gpu \
  model_name=deepseek2-16b \
  override_model_config=True \
  base_num_decoder_layers=16 \
  base_emb_dim=1024 \
  base_mlp_dim=2048 \
  base_moe_mlp_dim=1024 \
  num_experts=8 \
  num_experts_per_tok=2 \
  shared_experts=2 \
  first_num_dense_layers=1 \
  vocab_size=32000 \
  dataset_type=synthetic \
  reuse_example_batch=1 \
  per_device_batch_size=1 \
  max_target_length=256 \
  megablox=False \
  sparse_matmul=False \
  base_output_directory=/tmp/maxtext_runs \
  run_name=deepseek_moe_sub1b_gpu_v2 \
  enable_checkpointing=False \
  skip_jax_distributed_system=True \
  attention=dot_product \
  steps=50 \
  mu_dtype=bfloat16 \
  2>&1 | tee /tmp/training_log_deepseek_gpu_v2.txt

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
2026-06-18 15:41:52.700028: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[transformers] DeepseekV32Config got `key=rope_scaling` in kwargs but hasn't set it as attribute. For RoPE standardization you need to set `self.rope_parameters` in model's config. 
I0618 15:42:05.260700 135951648809600 max_utils.py:245] Skipping jax distributed system due to skip_jax_distributed_system=True flag.
I0618 15:42:05.351372   39282 pjrt_api.cc:119] 